### Structured Output

Models can be passed schemas to that they will use to structure their output. This is to ensure that the output can be easily parsed and used in subsequent processing. Langchain supports multiple schema types and methods for enforcing structured outputs.

#### init

In [22]:
import os
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.messages import HumanMessage

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:llama-3.3-70b-versatile")

#### Pydantic

Pydantic models provide the richest feature set with field validation, descriptions and nestes structures.

In [ ]:
from pydantic import BaseModel, Field

class Foodantic(BaseModel):
    """Structure that holds the info about a particular food"""
    name: str=Field(..., description="Name of the food")
    main_color: str=Field(description="The dominant color of the food")
    nutrient: str=Field(description="main nutrient in food")
    cost: float=Field(description="estimated buying or cooking cost")

agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    response_format=Foodantic
)

agent_response = agent.invoke({"messages": [HumanMessage("Provide me info about an italian meal")]})

model_w_output = model.with_structured_output(Foodantic)
model_response = model_w_output.invoke("Provide me info about an spanish meal")

print("Agent response: ")
print(agent_response["structured_response"])

print("\nModel response: ")
print(model_response)

Agent response: 
name='Pasta Carbonara' main_color='Yellow' nutrient='Carbohydrates' cost=15.0

Model response: 
name='Paella' main_color='Yellow' nutrient='Carbohydrates' cost=15.5


#### TypedDict

Useful if runtime validation is not necessary.

In [25]:
from typing import TypedDict, Annotated

class FoodDict(TypedDict):
    """Structure that holds the info about a particular food"""
    name: Annotated[str, ..., "name of the food"]
    main_color: Annotated[str, ...,"The dominant color of the food"]
    nutrient: Annotated[str, ..., "main nutrient in food"]
    cost: Annotated[float, ..., "estimated buying or cooking cost"]


agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    response_format=FoodDict
)

agent_response = agent.invoke({"messages": [HumanMessage("Provide me info about an italian meal")]})

model_w_output = model.with_structured_output(FoodDict)
model_response = model_w_output.invoke("Provide me info about an spanish meal")

print("Agent response: ")
print(agent_response["structured_response"])
print("\nModel response: ")
print(model_response)

Agent response: 
{'name': 'Spaghetti Bolognese', 'main_color': 'red', 'nutrient': 'carbohydrates', 'cost': 15.5}

Model response: 
{'cost': 15, 'main_color': 'Yellow', 'name': 'Paella', 'nutrient': 'Carbohydrates'}


#### @dataclass

The @dataclass decorator transforms a class it is declared before into structured output

In [26]:
from dataclasses import dataclass

@dataclass
class Foodclass:
    """Structure that holds the info about a particular food"""
    name: str
    main_color: str
    nutrient: str
    cost: float

agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    response_format=Foodclass
)

agent_response = agent.invoke({"messages": [HumanMessage("Provide me info about an italian meal")]})

model_w_output = model.with_structured_output(Foodclass)
model_response = model_w_output.invoke("Provide me info about an spanish meal")

print("Agent response: ")
print(agent_response["structured_response"])

print("\nModel response: ")
print(model_response)

Agent response: 
Foodclass(name='Pasta', main_color='Yellow', nutrient='Carbohydrates', cost=15.0)

Model response: 
{'cost': 15, 'main_color': 'Yellow', 'name': 'Paella', 'nutrient': 'Carbohydrates'}


#### Nested Classes and Mixed Module Utility

Classes can be nested and behave as types for attributes in other classes.
Field from Pydantic can be used with TypedDict and Annotate from TypedDict can be used with Pydantic.

In [35]:
class Ingredient(TypedDict):
    """Structure that holds the ingredient data of a food"""
    name: str=Field(..., description="Name of the ingredient")
    food_type: Annotated[str, ..., "category of food it falls under"]
    main_nutrient: str=Field(description="the main nutrient in the ingredient")

class FoodaticDict(BaseModel):
    """Structure that holds the info about a particular food"""
    name: str=Field(..., description="Name of the food")
    main_color: str=Field(description="The dominant color of the food")
    ingredients: list[Ingredient]=Field(description="main nutrient in food")
    cost: None | Annotated[str, ..., "estimated buying or cooking cost"]

model_w_output = model.with_structured_output(FoodaticDict)
agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    response_format=FoodaticDict
)

model_response = model_w_output.invoke("Provide details of a french meal")
messages = [
    HumanMessage("Provide the requested details of any italian meal")
]
agent_response = agent.invoke({"messages": messages})

print(f"Agent response: \n{agent_response["structured_response"]}")
print(f"\n Model Response: \n{model_response}")

Agent response: 
name='Spaghetti Bolognese' main_color='Red' ingredients=[{'name': 'Spaghetti', 'food_type': 'Carbohydrate', 'main_nutrient': 'Carbs'}, {'name': 'Minced Beef', 'food_type': 'Protein', 'main_nutrient': 'Protein'}, {'name': 'Tomato Sauce', 'food_type': 'Vegetable', 'main_nutrient': 'Vitamins'}] cost='15.00'

 Model Response: 
name='Coq au Vin' main_color='Brown' ingredients=[{'name': 'Chicken', 'food_type': 'Meat', 'main_nutrient': 'Protein'}, {'name': 'Mushrooms', 'food_type': 'Vegetable', 'main_nutrient': 'Fiber'}] cost='25'
